In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)
from src.tupl import run_tupl
from src.our_tupl import GENERATION_POLICIES, run_m1_tupl, run_all_senario_m1_tupl

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy
/home/asad/workspace/anaconda3/envs/asad/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DOMAIN_SET =['A','D','W']
DATA_DIR = './data/Office31/'
DATASET_DETAILS = {
    "prefix": 'office-',
    "suffix": '-resnet50-noft.mat',
    "resnet_feature": 'resnet50_features',
    "split_file_name": 'instanceSplit_office31_unseen15.mat',
}
NUM_LABELS=31

In [4]:
import json
import time
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/office31.json"
RESULT_CSV_PATH = "./result/csv/office31.csv"
RESULT_TIME_PATH = "./result/times/raw_time_office31.json"
RESULT_TIME_CSV_PATH = "./result/times/result_time_office31.csv"
path = Path(RESULT_OBJ_PATH)
time_path = Path(RESULT_TIME_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

if time_path.exists():
    with time_path.open("r", encoding="utf-8") as f:
        result_time = json.load(f)
else:
    result_time = {}

def save_results():
    time_path.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULT_OBJ_PATH, "w") as f:
        json.dump(result, f, indent=2)
    with open(RESULT_TIME_PATH, "w") as f:
        json.dump(result_time, f, indent=2)

result.keys(), result_time.keys()


(dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt']),
 dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt']))

In [5]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"
tupl = "TUPL"
our_tupl = "our_TUPL"

# clear last result
# _, _ = result.pop(base, None), result_time.pop(base, None)
# _, _ = result.pop(CCVAE, None), result_time.pop(CCVAE, None)
# _, _ = result.pop(our0, None), result_time.pop(our0, None)
# _, _ = result.pop(our_GRE, None), result_time.pop(our_GRE, None)
# _, _ = result.pop(tupl, None), result_time.pop(tupl, None)
# for k in [f"{our_tupl}_{p}" for p in GENERATION_POLICIES]:
#     _, _ = result.pop(k, None), result_time.pop(k, None)


## Base

In [6]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [7]:
if base not in result or base not in result_time:
    start = time.time()
    result[base] = run_all_senario(main_base, DOMAIN_SET)
    result_time[base] = time.time() - start
    save_results()


## GZSDA

In [8]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [9]:
if CCVAE not in result or CCVAE not in result_time:
    start = time.time()
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET)
    result_time[CCVAE] = time.time() - start
    save_results()


## m0

In [10]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [11]:
if our0 not in result or our0 not in result_time:
    start = time.time()
    result[our0] = run_all_senario(main_m0, DOMAIN_SET)
    result_time[our0] = time.time() - start
    save_results()


## m1: seperate after encoder

In [12]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [13]:
if our_GRE not in result or our_GRE not in result_time:
    start = time.time()
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET)
    result_time[our_GRE] = time.time() - start
    save_results()


## TUPL

In [14]:
def main_tupl(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_tupl(
        data_root="./data/",
        dataset="office31",
        source=args.sourceDomainIndex,
        target=args.targetDomainIndex,
        trial=args.trialIndex,
        seed=args.seed,
        device=device,
        quiet=True,
        return_model=False,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [15]:
if tupl not in result or tupl not in result_time:
    start = time.time()
    result[tupl] = run_all_senario(main_tupl, DOMAIN_SET)
    result_time[tupl] = time.time() - start
    save_results()


## our_TUPL: m1 VAE + TUPL

In [16]:
def main_m1_tupl(args, policy="real_plus_src2tgt"):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_m1_tupl(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        policy=policy,
        device=device,
        quiet=True,
        seed=args.seed,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [17]:
for p in GENERATION_POLICIES:
    key = f"{our_tupl}_{p}"
    if key not in result or key not in result_time:
        start = time.time()
        result.update(run_all_senario_m1_tupl(
            DOMAIN_SET=DOMAIN_SET,
            DATA_DIR=DATA_DIR,
            DATASET_DETAILS=DATASET_DETAILS,
            policies=[p],
        ))
        result_time[key] = time.time() - start
        save_results()


## Merge results

In [18]:
save_results()


In [19]:
# ignore our0
_, _ = result.pop(our0, None), result_time.pop(our0, None)
_, _ = result.pop("our_TUPL_real_plus_src2tgt", None), result_time.pop("our_TUPL_real_plus_src2tgt", None)
_, _ = result.pop("our_TUPL_real_plus_src2tgt_unseen", None), result_time.pop("our_TUPL_real_plus_src2tgt_unseen", None)
# _, _ = result.pop("our_TUPL_interp_src2tgt", None), result_time.pop("our_TUPL_interp_src2tgt", None)


In [20]:
import pandas as pd

n_senario = len(DOMAIN_SET) * (len(DOMAIN_SET) - 1)
n_trial = 5


df_time = pd.DataFrame(
    [
        {
            "method": method,
            "n_senario": n_senario,
            "n_trial": n_trial,
            "total_trial": n_senario * n_trial,
            "total_time": round(total_time, 1),
        }
        for method, total_time in result_time.items()
    ]
)
df_time.to_csv(RESULT_TIME_CSV_PATH, index=False)
# df_time


In [21]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]

# sort
df['method'] = pd.Categorical(
    df['method'],
    categories=[base, CCVAE, tupl, our0, our_GRE] + [f"{our_tupl}_{p}" for p in GENERATION_POLICIES],
    ordered=True,
)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,A -> D,base,92.32 ± 1.37,67.06 ± 4.20,77.30 ± 2.48
1,A -> D,CCVAE,87.73 ± 2.36,87.48 ± 2.46,87.42 ± 1.37
2,A -> D,TUPL,72.92 ± 1.90,64.69 ± 3.11,68.43 ± 2.32
3,A -> D,our_GRE,85.60 ± 2.53,85.66 ± 2.33,85.44 ± 1.39
4,A -> D,our_TUPL_interp_src2tgt,55.79 ± 3.63,57.22 ± 3.38,56.05 ± 2.59
5,A -> W,base,91.83 ± 0.44,58.19 ± 3.21,71.07 ± 2.47
6,A -> W,CCVAE,87.54 ± 1.09,82.67 ± 1.95,84.95 ± 0.95
7,A -> W,TUPL,69.83 ± 2.15,59.52 ± 2.95,63.89 ± 1.21
8,A -> W,our_GRE,85.19 ± 1.30,83.15 ± 1.38,84.10 ± 0.81
9,A -> W,our_TUPL_interp_src2tgt,59.71 ± 2.24,51.39 ± 2.95,54.87 ± 1.62


In [22]:
df.to_csv(RESULT_CSV_PATH, index=False)